In [35]:
import os
import re
import pandas as pd
from datetime import datetime
import sys

# --- Third-party libraries ---
try:
    from google.cloud import vision
    from google.cloud import language_v1
except ImportError:
    print("⚠️ WARNING: Google Cloud libraries not found. Please install them: pip install google-cloud-vision google-cloud-language")
    class vision: 
        ImageAnnotatorClient = lambda: None # type: ignore
        Image = lambda content: None # type: ignore
        ImageContext = lambda language_hints: None # type: ignore
        class types: 
            class Document: 
                Type = None 
                PLAIN_TEXT = "PLAIN_TEXT" 
                def __init__(self, content, type_, language): # type: ignore
                    self.content = content
                    self.type_ = type_
                    self.language = language
    class language_v1: 
        LanguageServiceClient = lambda: None # type: ignore
        class types: 
            class Document: 
                Type = None 
                PLAIN_TEXT = "PLAIN_TEXT" 
                def __init__(self, content, type_, language=None): # type: ignore
                    self.content = content
                    self.type_ = type_
                    self.language = language
            class EncodingType: 
                UTF8 = "UTF8" 

# --- Custom Hindu Calendar (from Gist) ---
_HinduCalendar = None; HINDU_CALENDAR_AVAILABLE = False
try:
    from hindu_calendar import HinduCalendar as HC 
    _HinduCalendar = HC; HINDU_CALENDAR_AVAILABLE = True; print("HinduCalendar module loaded successfully.")
except ImportError: print("⚠️ WARNING: hindu_calendar.py not found.")
except Exception as e: print(f"⚠️ WARNING: Error importing HinduCalendar: {e}")

# --- Constants and Patterns ---
_NAME_PART_REGEX = r"[A-Za-z\u0900-\u097F]+(?:-[A-Za-z\u0900-\u097F]+)*" 
_MULTI_WORD_NAME_REGEX = rf"{_NAME_PART_REGEX}(?:\s+{_NAME_PART_REGEX}){{0,2}}" 
NAME_CAPTURE_REGEX_FOR_PRIMARY_MARKERS = rf"{_MULTI_WORD_NAME_REGEX}(?:\s+स्त्री\s+{_MULTI_WORD_NAME_REGEX})?|{_NAME_PART_REGEX}स्त्री{_MULTI_WORD_NAME_REGEX}|{_MULTI_WORD_NAME_REGEX}"

PRIMARY_PERSON_MARKERS_LIST_UNSORTED = [
    "प्राग साये", "प्राग", "आदि", "प्रा०मु०", "प्रा0मु0", "प्रा०", "प्रा0", 
    "प्रत्", "प्रगोकल", "प्र.", "प्र0", "प्र。", "ठाο", "ठाग्र", "डा0", "मो0", 
    "ठाप्पारधी प्र。", "गणससोदिया नामदेव मोण्जीवालीगंज हैपी रोड प्र", "गव्वयेले प्र",
    "• डा0तोमर मो0 जीवाजीगंज प्र0", 
    "'ठाकढवाहे मोठका का साहब कानिवाल गेट प्र0", 
    "••मोनजनकगंज प्र", 
    "* ठाग्रराठौर प्र0", 
    "'बछेले गोठेंगर प्र0", 
    "'बघेले टेंगर प्र",
    "बजेटले गो0 ठेंगर प्र.",
    "प्रारोज" 
]
PRIMARY_PERSON_MARKERS_LIST = sorted(list(set(PRIMARY_PERSON_MARKERS_LIST_UNSORTED)), key=len, reverse=True)
PRIMARY_PERSON_MARKERS_REGEX_STR = "|".join(re.escape(m.lstrip("'\"• ")) for m in PRIMARY_PERSON_MARKERS_LIST) 
PRIMARY_PERSON_MARKERS = re.compile(rf"({PRIMARY_PERSON_MARKERS_REGEX_STR})\s*({NAME_CAPTURE_REGEX_FOR_PRIMARY_MARKERS})")
PRIMARY_PERSON_SELF_RELATION = re.compile(rf"({PRIMARY_PERSON_MARKERS_REGEX_STR})\s*({NAME_CAPTURE_REGEX_FOR_PRIMARY_MARKERS})\s*(की|का|के)\s*(पतोह(?:\u200c)?|पतोहू(?:\u200c)?|लड़(?:\u200c)?का|लडका|बेटी|लड़की|पत्नी)(?!\s+{_MULTI_WORD_NAME_REGEX})")
IMPLICIT_PRIMARY_BETA_POTA = re.compile(rf"^({_MULTI_WORD_NAME_REGEX}(?:\sशर्मा)?)\s+(बेटा|पोता|नाती)\s+({_MULTI_WORD_NAME_REGEX})\s*(?:के)?(?=\s+|$)") 
IMPLICIT_PRIMARY_STARTING_WITH_BETA = re.compile(rf"^बेटा\s+({_MULTI_WORD_NAME_REGEX})\s+पिता\s*({_MULTI_WORD_NAME_REGEX})")

SELF_RELATION_TO_ANCESTOR_PATTERNS = [
    ("पिता", re.compile(rf"बेट\s*({_MULTI_WORD_NAME_REGEX})(?:\s*कि)?")), 
    ("दादा", re.compile(rf"पोता\s*({_MULTI_WORD_NAME_REGEX})(?:\s*कि)?")), 
    ("परदादा", re.compile(rf"^पो-\s*({_MULTI_WORD_NAME_REGEX})(?:\s*कि)?")),
    ("पिता", re.compile(rf"^(नदीक\s+व)\s*लड़(?:\u200c)?का")), 
]
OWNER_OF_RELATION_PATTERNS = [ 
    ("लड़का", re.compile(rf"({_MULTI_WORD_NAME_REGEX})\s+की\s+(?:लड़(?:\u200c)?का|लडका)\s*(.+)")) , 
    ("बेटी", re.compile(rf"({_MULTI_WORD_NAME_REGEX})\s+की\s+(?:बेटी|लड़की)\s*(.+)")), 
    ("पत्नी", re.compile(rf"({_MULTI_WORD_NAME_REGEX})\s+की\s+पत्नी\s*({_MULTI_WORD_NAME_REGEX})")), 
    ("पतोह", re.compile(rf"({_MULTI_WORD_NAME_REGEX})\s+की\s+(?:पतोह(?:\u200c)?|पतोहू(?:\u200c)?)\s*({_MULTI_WORD_NAME_REGEX})")), 
]
_NAME_LIST_CAPTURE_REGEX = r"(.+)" 
IMPLIED_SUBJECT_RELATIONS = [ 
    ("भाई", re.compile(r"^के\s+भाई\s*" + _NAME_LIST_CAPTURE_REGEX)), 
    ("लड़का", re.compile(r"^के\s+लड़का\s*" + _NAME_LIST_CAPTURE_REGEX)), 
    ("बेटा", re.compile(r"^के\s+बेटा\s*" + _NAME_LIST_CAPTURE_REGEX)),   
    ("पिता", re.compile(r"^के\s+पिता\s*({_MULTI_WORD_NAME_REGEX})")), 
    ("दादा", re.compile(r"^के\s+दादा\s*({_MULTI_WORD_NAME_REGEX})")), 
    ("दादा", re.compile(r"^के\s+पोता\s*({_MULTI_WORD_NAME_REGEX})")), 
    ("बेटी", re.compile(r"^की\s+बेटी\s*" + _NAME_LIST_CAPTURE_REGEX)), 
    ("लड़की", re.compile(r"^की\s+लड़की\s*" + _NAME_LIST_CAPTURE_REGEX)), 
    ("पत्नी", re.compile(r"^की\s+पत्नी\s*({_MULTI_WORD_NAME_REGEX})")),
    ("पतोह", re.compile(r"^की\s+(?:पतोह(?:\u200c)?|पतोहू(?:\u200c)?)\s*({_MULTI_WORD_NAME_REGEX})")),
    ("माता", re.compile(r"^के\s+साये\s+माता\s*({_MULTI_WORD_NAME_REGEX})")), 
    ("मामा", re.compile(r"^के\s+मामा\s*({_MULTI_WORD_NAME_REGEX})")), 
]
IMPLIED_SUBJECT_DIRECT_RELATIONS = [ 
    ("भाई", re.compile(r"^भाई\s*(.+)")), 
    ("लड़का", re.compile(r"^लड़का\s*(.+)")),
    ("बेटा", re.compile(r"^बेटा\s*(.+)")),
    ("पिता", re.compile(r"^पिता\s*({_MULTI_WORD_NAME_REGEX})")),
    ("दादा", re.compile(r"^दादा\s*({_MULTI_WORD_NAME_REGEX})")),
    ("पोता", re.compile(r"^पोता\s*({_MULTI_WORD_NAME_REGEX})")),
    ("बेटी", re.compile(r"^बेटी\s*(.+)")),
    ("लड़की", re.compile(r"^लड़की\s*(.+)")),
    ("पत्नी", re.compile(r"^पत्नी\s*({_MULTI_WORD_NAME_REGEX})")),
    ("पतोह", re.compile(r"^पतोह(?:\u200c)?\s*({_MULTI_WORD_NAME_REGEX})")),
    ("पतोहू", re.compile(r"^पतोहू(?:\u200c)?\s*({_MULTI_WORD_NAME_REGEX})")),
    ("भतीजा", re.compile(r"^भतीजा\s*(.+)")),
    ("माता", re.compile(r"^माता\s*({_MULTI_WORD_NAME_REGEX})")), 
    ("मामा", re.compile(r"^मामा\s*({_MULTI_WORD_NAME_REGEX})")), 
]
OWNER_KE_RELATION_PATTERNS = [ 
    ("भाई", re.compile(rf"^({_MULTI_WORD_NAME_REGEX})\s+के\s+भाई\s*(.+)")),
    ("लड़का", re.compile(rf"^({_MULTI_WORD_NAME_REGEX})\s+के\s+लड़का\s*(.+)")),
    ("बेटा", re.compile(rf"^({_MULTI_WORD_NAME_REGEX})\s+के\s+बेटा\s*(.+)")),
    ("पिता", re.compile(rf"^({_MULTI_WORD_NAME_REGEX})\s+के\s+पिता\s*({_MULTI_WORD_NAME_REGEX})")),
    ("पोता", re.compile(rf"^({_MULTI_WORD_NAME_REGEX})\s+के\s+पोता\s*({_MULTI_WORD_NAME_REGEX})")), 
]
EXPLICIT_PAIR_RELATIONS = [ 
    ("लड़का", re.compile(rf"^({_MULTI_WORD_NAME_REGEX})\s+(?:लड़(?:\u200c)?का|लडका)\s+({_MULTI_WORD_NAME_REGEX})(?=\s+|$)")), 
    ("बेटा", re.compile(rf"^({_MULTI_WORD_NAME_REGEX})\s+बेटा\s+({_MULTI_WORD_NAME_REGEX})(?=\s+|$)")),
    ("भाई", re.compile(rf"^({_MULTI_WORD_NAME_REGEX})\s+भाई\s+({_MULTI_WORD_NAME_REGEX})(?=\s+|$)")),
]
INVALID_NAME_PARTICLES_AS_NAMES = ["के", "की", "का", "व", "और", "तथा", "साये", "नाती", 
                                   "पिता", "लड़का", "बेटा", "भाई", "दादा", "पोता", 
                                   "माता", "मामा", "पत्नी", "पतोह", "भतीजा", "संग", "संगे", "साथै", "साधै",
                                   "वाल", "वाला", "वाले", "जी"] 
KNOWN_CASTE_WORDS = ["राजपूत", "राजावत", "ब्राह्मण", "शुक्ला", "पाठक", "जोशी", "ठाकुर", "गुजर", "यादव", "बघेल", "शर्मा", "वर्मा"]

def split_conjoined_names(name_chunk_str):
    if not name_chunk_str: return []
    normalized_chunk = re.sub(r"[\s,।;/]+(?:और|तथा)?[\s,।;/]*", " व ", name_chunk_str) 
    normalized_chunk = re.sub(r'\s*व\s*', ' व ', normalized_chunk.strip())
    names_found = []
    potential_names = normalized_chunk.split(' व ')
    for name_part in potential_names:
        cleaned_part_initial = clean_extracted_name(name_part.strip())
        if not cleaned_part_initial: continue

        final_name_to_add = cleaned_part_initial
        for rel_word_check in ["लड़का", "बेटा", "भाई", "नाती", "पोता", "भतीजा"]: 
            if cleaned_part_initial.startswith(rel_word_check + " "):
                possible_name_after_rel = cleaned_part_initial[len(rel_word_check)+1:].strip()
                if possible_name_after_rel and len(clean_extracted_name(possible_name_after_rel)) >= 2 and \
                   clean_extracted_name(possible_name_after_rel).lower() not in INVALID_NAME_PARTICLES_AS_NAMES:
                    final_name_to_add = clean_extracted_name(possible_name_after_rel)
                    break 
        
        if final_name_to_add and len(final_name_to_add) >= 2 and \
           final_name_to_add.lower() not in INVALID_NAME_PARTICLES_AS_NAMES and \
           not final_name_to_add.isdigit() and \
           not (len(final_name_to_add) == 1 and not final_name_to_add.isalpha()): 
            names_found.append(final_name_to_add)
        elif name_part.strip() and not (final_name_to_add and final_name_to_add.lower() in INVALID_NAME_PARTICLES_AS_NAMES) :
            pass
    return names_found

NAME_CONJUNCTION_PATTERN = re.compile(rf"({_MULTI_WORD_NAME_REGEX})\s+व\s*({_MULTI_WORD_NAME_REGEX})") 
CONJUNCTION_WITH_STRI = re.compile(rf"({_MULTI_WORD_NAME_REGEX})\s+व\s*({_MULTI_WORD_NAME_REGEX})\s+साथ में स्त्री(?:-|.)?")
SANG_ME_STRI_PATTERN = re.compile(r"संग में स्त्री(?:-|.)?") 
HINDI_MONTH_MAP = { "चैत": "चैत्र", "चैत्र": "चैत्र", "चे": "चैत्र", "बैसाख": "वैशाख", "वैशाख": "वैशाख", "बेशाख": "वैशाख", "बेसाख": "वैशाख", "बैसाष": "वैशाख","जेठ": "ज्येष्ठ", "ज्येष्ठ": "ज्येष्ठ","असाढ़": "आषाढ़", "आषाढ़": "आषाढ़", "अषाढ": "आषाढ़","सावन": "श्रावण", "श्रावण": "श्रावण", "श्रवण": "श्रावण","भादो": "भाद्रपद", "भादों": "भाद्रपद", "भाद्रपद": "भाद्रपद", "भादवा": "भाद्रपद","अश्विन": "आश्विन", "आसिन": "आश्विन", "कुआर": "आश्विन", "क्वार": "आश्विन", "कातिक": "कार्तिक", "कार्तिक": "कार्तिक", "काती": "कार्तिक","अगहन": "मार्गशीर्ष", "मंगसर": "मार्गशीर्ष", "मार्गशीर्ष": "मार्गशीर्ष", "अघन": "मार्गशीर्ष","पूस": "पौष", "पौष": "पौष", "पोस": "पौष","माघ": "माघ", "माह": "माघ", "माय": "माघ","फागुन": "फाल्गुन", "फाल्गुन": "फाल्गुन", "फगुन": "फाल्गुन", "फागण": "फाल्गुन","मिती": None, "मिटी": None, "प्रमाठ": None, "चादव": None, "सम्माङ": None, }
HINDI_MONTH_TO_NUMBER = { "चैत्र": 1, "वैशाख": 2, "ज्येष्ठ": 3, "आषाढ़": 4, "श्रावण": 5, "भाद्रपद": 6, "आश्विन": 7, "कार्तिक": 8, "मार्गशीर्ष": 9, "पौष": 10, "माघ": 11, "फाल्गुन": 12, }
HINDI_MONTH_NAMES_FOR_REGEX = "|".join(re.escape(k) for k in HINDI_MONTH_MAP.keys() if HINDI_MONTH_MAP[k] is not None) 
STANDARD_DATE_REGEX = re.compile(rf"(?:मिती|मिटी\s*)?\s*(?P<month_name>{HINDI_MONTH_NAMES_FOR_REGEX})\s*(?:(?P<paksha>सुदी|वदी|बदी)\s*(?P<day>\d{{1,2}}))?\s*(?:(?:सम्बत्?|सम्बर|सं\.?)\s*(?P<year>\d{{4}}))?")
PAKSHA_TITHI_REGEX = re.compile(r"^(सुदी|वदी|बदी)\s*(\d{1,2}|च|छ)") 
DATE_DDMM_YYYY_FORMAT_REGEX = re.compile(r"(?:दिनांक\s*)?(?P<day>\d{2})(?P<month>\d{2})/(?P<year>\d{4})")
DATE_DD_MM_YY_FORMAT_REGEX = re.compile(r"(?P<day>\d{1,2})[-./](?P<month>\d{1,2})[-./](?P<year>\d{2}(?:\d{2})?)") 

def detect_text(path):
    if not os.getenv('GOOGLE_APPLICATION_CREDENTIALS'): print("⚠️ WARNING: GOOGLE_APPLICATION_CREDENTIALS not set. Vision API call will be skipped."); return []
    if vision.ImageAnnotatorClient is None or not callable(vision.ImageAnnotatorClient): print("⚠️ WARNING: Vision API client not available. Skipping text detection."); return [] # type: ignore
    client = vision.ImageAnnotatorClient(); _image = vision.Image(content=open(path, "rb").read()) # type: ignore
    response = client.document_text_detection(image=_image, image_context=vision.ImageContext(language_hints=["hi"])) # type: ignore
    if hasattr(response, 'error') and response.error and response.error.message: raise Exception(f"Vision API Error: {response.error.message}")
    return [t.description for t in response.text_annotations] if hasattr(response, 'text_annotations') and response.text_annotations else []

def clean_text(raw_text): 
    text = raw_text.replace('\r', ''); text = text.replace('\n', '###NEWLINE###') 
    text = re.sub(r'[ \t]+', ' ', text).strip(); transliteration_table = str.maketrans("०१२३४५६७८९", "0123456789"); return text.translate(transliteration_table)

def analyze_text_entities(text_content, language_client):
    text_for_nlp = text_content.replace("###NEWLINE###", " ").strip()
    if not text_for_nlp or not language_client: return []
    if not os.getenv('GOOGLE_APPLICATION_CREDENTIALS'): print("⚠️ WARNING: GOOGLE_APPLICATION_CREDENTIALS not set. NLP API will likely fail or be skipped."); return []
    if language_v1.LanguageServiceClient is None or not callable(language_v1.LanguageServiceClient): print("⚠️ WARNING: Language API client not available. Skipping entity analysis."); return [] # type: ignore
    document = language_v1.types.Document(content=text_for_nlp, type_=language_v1.types.Document.Type.PLAIN_TEXT, language="hi") # type: ignore
    encoding_type = language_v1.types.EncodingType.UTF8 # type: ignore
    try: response = language_client.analyze_entities(request={"document": document, "encoding_type": encoding_type}); return response.entities if hasattr(response, 'entities') else []
    except Exception as e: print(f"Error during NLP entity analysis (lang='hi'): {e}"); return []

def standardize_month_name(month_str):
    if not month_str: return None; month_str_cleaned = re.sub(r"(सुदी|वदी|बदी)$", "", month_str.strip()).strip(); return HINDI_MONTH_MAP.get(month_str_cleaned)

def convert_hindu_to_gregorian_date(hc_instance, day_str, month_str, year_str):
    if not HINDU_CALENDAR_AVAILABLE or not hc_instance: return None;  _ = paksha_from_month = "" # type: ignore
    if not all([day_str, month_str, year_str]): return None 
    try: day = int(day_str); vs_year = int(year_str)
    except ValueError: return None
    if " सुदी" in month_str: paksha_from_month = "सुदी"; month_str = month_str.replace(" सुदी", "").strip()
    elif " वदी" in month_str: paksha_from_month = "वदी"; month_str = month_str.replace(" वदी", "").strip()
    elif " बदी" in month_str: paksha_from_month = "बदी"; month_str = month_str.replace(" बदी", "").strip()
    standard_month = standardize_month_name(month_str)
    if not standard_month: return None
    hindu_month_number = HINDI_MONTH_TO_NUMBER.get(standard_month)
    if not hindu_month_number: return None
    if not (1 <= day <= 32) or not (1000 < vs_year < 3000): return None 
    hindu_date_input_str = f"{day}/{hindu_month_number}/{vs_year}"
    try:
        date_object = hc_instance.find_regional_date(hindu_date_input_str) 
        if date_object and 'ce_date' in date_object and date_object['ce_date']:
            return datetime.strptime(date_object['ce_date'], "%d/%m/%Y").strftime("%Y-%m-%d")
    except Exception as e: print(f"DEBUG: HinduCalendar conversion error for '{hindu_date_input_str}': {e}"); pass 
    return None

def clean_extracted_name(name_raw):
    if not name_raw: return ""
    name = name_raw.strip()
    if name.endswith(" के साये"): name = name[:-len(" के साये")].strip()
    suffixes_to_remove = [" की", " के", " का", " कि", " कु"] 
    for suffix in suffixes_to_remove:
        if name.endswith(suffix): name = name[:-len(suffix)].strip()
    if name.endswith("के") and len(name) > 1 : 
        name_prefix = name[:-1]
        if re.fullmatch(_MULTI_WORD_NAME_REGEX, name_prefix) or name_prefix.endswith("सिंह") or name_prefix.endswith("सिह"): 
            name = name_prefix
    name = re.sub(r"^\s*[^A-Za-z\u0900-\u097F\-(\[]+", "", name, flags=re.UNICODE) 
    name = re.sub(r"[^A-Za-z\u0900-\u097F\s\-()]+$", "", name, flags=re.UNICODE) 
    return name.strip("(). ")

def tokenize_complex_name(name_chunk, original_ocr_snippet=""):
    return [name_chunk] 

# --- Main Processing Function ---
def process_document_structure(cleaned_full_text_with_markers, language_client, hc_instance, file_name="Unknown_File"):
    all_extracted_records = []
    overall_individual_id_counter = 1
    bahi_number_global = ""
    m_bahi_global = re.search(r'(बही|वही)\s*([^\s]*(?:की)?)\s*(?:नम्बर|नं\.?|No\.?|नम्वर)\s*([^\s]+)', cleaned_full_text_with_markers)
    if m_bahi_global:
        bahi_name_part = m_bahi_global.group(2).strip()
        bahi_num_val = m_bahi_global.group(3).strip()
        bahi_number_global = f"{m_bahi_global.group(1)} {bahi_name_part} {bahi_num_val}".replace("  ", " ").strip()

    record_blocks = cleaned_full_text_with_markers.split('###NEWLINE###')
    paragraphs = [block.strip() for block in record_blocks if block.strip()]
    if not paragraphs and cleaned_full_text_with_markers.strip(): 
        paragraphs.append(cleaned_full_text_with_markers.replace("###NEWLINE###"," ").strip())
    print(f"Segmented into {len(paragraphs)} paragraphs (based on newlines).")

    # Block context variables - these are reset when a new primary person is definitively identified
    current_block_caste, current_block_subcaste, current_block_place = "", "", ""
    current_block_primary_person = "" 
    current_block_primary_is_female = False
    current_block_primary_relation_desc = "" 
    current_block_husband_name = None 
    current_block_wife_name = None    
    current_block_father_name_of_primary = None 
    current_block_grandfather_name_of_primary = None
    current_block_father_in_law = None 
    current_block_date_text, current_block_date_gregorian = "", ""
    current_block_folio_number = "" 
    temp_block_month, temp_block_paksha, temp_block_day, temp_block_year = None, None, None, None
    
    initial_folio_processed = False
    if len(paragraphs) > 0 and re.fullmatch(r"\d+", paragraphs[0].strip()) and len(paragraphs[0].strip()) < 4:
        current_block_folio_number = paragraphs[0].strip()
        print(f"  CONTEXT SET (Pre-loop): Folio Number from P1: '{current_block_folio_number}'")
        paragraphs[0] = "" 
        initial_folio_processed = True
    
    start_para_index_for_csp = 0 if not initial_folio_processed else 1
    if len(paragraphs) > start_para_index_for_csp: 
        csp_candidate_text = paragraphs[start_para_index_for_csp].lstrip("• ").strip().strip("'\"")
        csp_candidate_text = re.sub(r"^\s*(?:\d+\.\s*|\d+\s+)", "", csp_candidate_text).strip()
        m_csp1 = re.match(r'^\s*(?P<caste>[^\s(]+)\s*\((?P<subcaste>[^)]+)\)\s*वासी\s*(?P<place_full>.+?)(?=\s*(?:लौहार\sविश्वकर्मा\s*\)|प्रा0|प्रा०|जात|$|,|###NEWLINE###))', csp_candidate_text, re.IGNORECASE)
        m_csp2 = None
        if not m_csp1: m_csp1 = re.match(r'^\s*(?P<caste>[^\s(]+)\s*\((?P<subcaste>[^)]+)\)\s*वासी\s*(?P<place_full>[^\s,।;:-]+(?:(?:\s+[^\s,।;:-]+)*))', csp_candidate_text, re.IGNORECASE)
        if not m_csp1: 
            m_csp2 = re.match(r'^\s*(?P<caste>[A-Za-z\u0900-\u097F]+(?:\s+[A-Za-z\u0900-\u097F]+)?)\s+वासी\s+(?P<place_full>[^\s,।;:-]+(?:(?:\s+[^\s,।;:-]+)*))(?=\s*(?:प्रा0|प्रा०|जात|$|,|###NEWLINE###))', csp_candidate_text, re.IGNORECASE)
            if not m_csp2: m_csp2 = re.match(r'^\s*(?P<caste>[A-Za-z\u0900-\u097F]+(?:\s+[A-Za-z\u0900-\u097F]+)?)\s+वासी\s+(?P<place_full>[^\s,।;:-]+(?:(?:\s+[^\s,।;:-]+)*))', csp_candidate_text, re.IGNORECASE)
        m_csp_active_header = m_csp1 or m_csp2
        if m_csp_active_header:
            current_block_caste = m_csp_active_header.group('caste').strip()
            current_block_subcaste = m_csp_active_header.group('subcaste').strip() if 'subcaste' in m_csp_active_header.groupdict() and m_csp_active_header.group('subcaste') else ""
            current_block_place = m_csp_active_header.group('place_full').strip().rstrip(' के')
            print(f"  CONTEXT SET (Pre-loop CSP): Caste='{current_block_caste}', Subcaste='{current_block_subcaste}', Place='{current_block_place}' from P{start_para_index_for_csp+1}-like structure.")
            if csp_candidate_text == m_csp_active_header.group(0) and \
               not (PRIMARY_PERSON_MARKERS.search(paragraphs[start_para_index_for_csp][m_csp_active_header.end():]) or \
                    IMPLICIT_PRIMARY_BETA_POTA.match(paragraphs[start_para_index_for_csp][m_csp_active_header.end():].strip())):
                paragraphs[start_para_index_for_csp] = "" 

    for para_idx, para_text_original in enumerate(paragraphs):
        if not para_text_original.strip(): continue
        print(f"\n--- Processing Paragraph {para_idx + 1} ('{para_text_original[:70]}...') ---")
        
        para_data_position = str(cleaned_full_text_with_markers.find(para_text_original)) 
        family_id = f"F{para_idx + 1:03d}" 
        line_folio_number_temp = "" 
        individuals_for_this_line = []
        added_names_on_this_line = set() 
        
        text_to_parse_for_current_line_raw = para_text_original.lstrip("• ").strip() 
        text_to_parse_for_current_line = re.sub(r"^\s*(?:\d+\.\s*|\d+\s+)", "", text_to_parse_for_current_line_raw).strip()
        text_to_parse_for_current_line = text_to_parse_for_current_line.strip("'\"") 
        
        print(f"    Text after initial strip: '{text_to_parse_for_current_line[:70]}'")
        
        line_caste, line_subcaste, line_place = current_block_caste, current_block_subcaste, current_block_place 
        leading_caste_on_line = "" # INITIALIZE HERE

        temp_text_for_caste_check = text_to_parse_for_current_line
        if temp_text_for_caste_check.startswith("जात "):
            m_jat = re.match(r'जात\s+([^\s]+)(?:\s+([^\sप्रागसायेगांवजिलामितीलड़काबेटीभाईस्त्री]+))?', temp_text_for_caste_check)
            if m_jat:
                line_caste = m_jat.group(1).strip(); line_subcaste = m_jat.group(2).strip() if m_jat.group(2) else ""
                text_to_parse_for_current_line = temp_text_for_caste_check[m_jat.end():].strip()
                print(f"  Line Context (जात): Caste='{line_caste}', Subcaste='{line_subcaste}'")
        else:
            known_castes_prefixes_markers = ["ग्वाला प्रा", "लोहार प्रा", "सुडियार प्रा", "टेलीमहती प्रा", "टेलीमहतो प्रा", "टेली साहू प्रा", "टेलीसाडू प्रा"] 
            for known_prefix in known_castes_prefixes_markers:
                if temp_text_for_caste_check.startswith(known_prefix):
                    caste_part = known_prefix.split(" प्रा")[0]
                    leading_caste_on_line = caste_part 
                    text_to_parse_for_current_line = "प्रा" + temp_text_for_caste_check[len(known_prefix):] 
                    line_caste = caste_part 
                    print(f"    Leading caste prefix '{caste_part}' found. Line Caste set to: '{line_caste}'. New text: '{text_to_parse_for_current_line[:50]}'")
                    break
        
        print(f"    Text after line-specific context processing: '{text_to_parse_for_current_line[:70]}'")
        
        raw_primary_consumed_text = ""; temp_line_pp_name_candidate = None; temp_line_pp_desc = None
        line_has_primary_marker = False; is_implicit_primary = False
        text_after_primary_phrase_on_line = text_to_parse_for_current_line 
        identified_father_for_primary = None; identified_grandfather_for_primary = None
        text_before_primary_marker_on_this_line = ""
        
        m_primary_search = PRIMARY_PERSON_MARKERS.search(text_to_parse_for_current_line)
        m_primary_self_rel_search = PRIMARY_PERSON_SELF_RELATION.search(text_to_parse_for_current_line)
        actual_marker_match = None

        if m_primary_self_rel_search and m_primary_search:
            actual_marker_match = m_primary_self_rel_search if m_primary_self_rel_search.start() <= m_primary_search.start() else m_primary_search
        elif m_primary_self_rel_search: actual_marker_match = m_primary_self_rel_search
        elif m_primary_search: actual_marker_match = m_primary_search

        if actual_marker_match:
            line_has_primary_marker = True; is_implicit_primary = False
            raw_primary_consumed_text = actual_marker_match.group(0) 
            temp_line_pp_name_candidate = clean_extracted_name(actual_marker_match.group(2).strip())
            if actual_marker_match == m_primary_self_rel_search:
                temp_line_pp_desc = actual_marker_match.group(4).strip().replace("\u200c", "")
            print(f"    Primary marker strategy: EXPLICIT. Marker: '{actual_marker_match.group(1)}', Raw consumed: '{raw_primary_consumed_text}', Name Candidate: '{temp_line_pp_name_candidate}'")
        
        starts_with_relation_prefix = text_to_parse_for_current_line.startswith("के ") or \
                                     text_to_parse_for_current_line.startswith("की ") or \
                                     any(text_to_parse_for_current_line.startswith(rel_word + " ") for rel_word in ["भाई", "लड़का", "बेटा", "पिता", "दादा", "पोता", "माता", "मामा", "पत्नी", "पतोह", "भतीजा"])
        
        try_implicit_condition = (not line_has_primary_marker and not starts_with_relation_prefix) or \
                                 (actual_marker_match and actual_marker_match.start() > 0 and not starts_with_relation_prefix) or \
                                 (not current_block_primary_person and not starts_with_relation_prefix) 

        if try_implicit_condition:
            prefix_to_check_implicit = text_to_parse_for_current_line
            if actual_marker_match and actual_marker_match.start() > 0 :
                 prefix_to_check_implicit = text_to_parse_for_current_line[:actual_marker_match.start()].strip()
            
            m_implicit_primary = IMPLICIT_PRIMARY_BETA_POTA.match(prefix_to_check_implicit)
            if not m_implicit_primary and not prefix_to_check_implicit: 
                m_implicit_primary = IMPLICIT_PRIMARY_BETA_POTA.match(text_to_parse_for_current_line)
            
            if m_implicit_primary and (not line_has_primary_marker or (actual_marker_match and m_implicit_primary.start(0) < actual_marker_match.start())):
                print(f"    Primary marker strategy: IMPLICIT preferred/found. Raw consumed: '{m_implicit_primary.group(0)}'")
                temp_line_pp_name_candidate = clean_extracted_name(m_implicit_primary.group(1))
                relation_word = m_implicit_primary.group(2) 
                ancestor_name = clean_extracted_name(m_implicit_primary.group(3))
                
                # Check if this implicit primary is better than a later explicit one
                if line_has_primary_marker and actual_marker_match and m_implicit_primary.start(0) >= actual_marker_match.start():
                    print("      Explicit marker found earlier or at same position, preferring explicit.")
                else:
                    raw_primary_consumed_text = m_implicit_primary.group(0) 
                    line_has_primary_marker = True; is_implicit_primary = True; temp_line_pp_desc = ""
                    if relation_word == "बेटा": identified_father_for_primary = ancestor_name
                    elif relation_word in ["पोता", "नाती"]: identified_grandfather_for_primary = ancestor_name
                    
                    implicit_match_start_in_original_line = text_to_parse_for_current_line.find(raw_primary_consumed_text)
                    if implicit_match_start_in_original_line != -1:
                        text_before_primary_marker_on_this_line = text_to_parse_for_current_line[:implicit_match_start_in_original_line].strip()
                    else: text_before_primary_marker_on_this_line = ""


        if line_has_primary_marker and temp_line_pp_name_candidate:
            print(f"    --- NEW PRIMARY BLOCK for '{temp_line_pp_name_candidate}' (was: '{current_block_primary_person}') ---")
            _preserved_folio = current_block_folio_number 
            _previous_block_caste, _previous_block_subcaste, _previous_block_place = current_block_caste, current_block_subcaste, current_block_place
            
            current_block_primary_person = "" ; current_block_primary_is_female = False
            current_block_primary_relation_desc = temp_line_pp_desc or ""
            current_block_husband_name = None; current_block_wife_name = None    
            current_block_father_name_of_primary = identified_father_for_primary 
            current_block_grandfather_name_of_primary = identified_grandfather_for_primary
            current_block_father_in_law = None 
            current_block_date_text, current_block_date_gregorian = "", ""
            temp_block_month, temp_block_paksha, temp_block_day, temp_block_year = None, None, None, None
            current_block_folio_number = _preserved_folio 
            
            # Determine context (Caste, Subcaste, Place) for this new block's records
            # If line_caste/subcaste/place were set by जात or leading_caste_on_line, they are already set
            # Otherwise, try to get from text_before_primary_marker_on_this_line
            if not (line_caste or line_subcaste or line_place): # If not set by जात or leading_caste
                if text_before_primary_marker_on_this_line:
                    print(f"    INFO: Text found before primary marker on this line: '{text_before_primary_marker_on_this_line}'")
                    potential_caste_from_prefix = ""
                    for kc in KNOWN_CASTE_WORDS:
                        if text_before_primary_marker_on_this_line.startswith(kc): 
                            potential_caste_from_prefix = kc
                            line_caste = kc
                            line_place = text_before_primary_marker_on_this_line[len(kc):].strip().lstrip("- ")
                            break
                    if not potential_caste_from_prefix: 
                        line_place = text_before_primary_marker_on_this_line
                    print(f"      CONTEXT from prefix: Caste='{line_caste}', Place='{line_place}'")
                else: # Inherit from previous block if no prefix and no line-specific caste info
                    line_caste, line_subcaste, line_place = _previous_block_caste, _previous_block_subcaste, _previous_block_place
            
            current_block_caste, current_block_subcaste, current_block_place = line_caste, line_subcaste, line_place
            print(f"    Block Context Updated/Set: Caste='{current_block_caste}', Subcaste='{current_block_subcaste}', Place='{current_block_place}'")

            match_start_index_final = text_to_parse_for_current_line.find(raw_primary_consumed_text)
            if match_start_index_final != -1:
                text_after_primary_phrase_on_line = text_to_parse_for_current_line[match_start_index_final + len(raw_primary_consumed_text):].strip()
            else: text_after_primary_phrase_on_line = text_to_parse_for_current_line.replace(raw_primary_consumed_text, "", 1).strip()
            print(f"    Text after primary marker phrase (for iterative loop): '{text_after_primary_phrase_on_line[:50]}'")

            if not current_block_father_name_of_primary and not is_implicit_primary: 
                m_name_beta_father = re.match(rf"^({_MULTI_WORD_NAME_REGEX})\s+बेटा\s*({_MULTI_WORD_NAME_REGEX})$", temp_line_pp_name_candidate)
                if m_name_beta_father:
                    refined_pp_name = clean_extracted_name(m_name_beta_father.group(1))
                    current_block_father_name_of_primary = clean_extracted_name(m_name_beta_father.group(2))
                    temp_line_pp_name_candidate = refined_pp_name
            
            primary_name_phrase = temp_line_pp_name_candidate
            m_name_stri_husband = re.match(rf"^({_MULTI_WORD_NAME_REGEX})\s+स्त्री\s+({_MULTI_WORD_NAME_REGEX})$", primary_name_phrase)
            if m_name_stri_husband:
                current_block_primary_person = clean_extracted_name(m_name_stri_husband.group(1))
                current_block_husband_name = clean_extracted_name(m_name_stri_husband.group(2))
                current_block_wife_name = current_block_primary_person; current_block_primary_is_female = True
            else:
                m_namestri_husband_nospace = re.match(rf"^({_NAME_PART_REGEX})स्त्री({_MULTI_WORD_NAME_REGEX})$", primary_name_phrase)
                if m_namestri_husband_nospace:
                    current_block_primary_person = clean_extracted_name(m_namestri_husband_nospace.group(1))
                    current_block_husband_name = clean_extracted_name(m_namestri_husband_nospace.group(2))
                    current_block_wife_name = current_block_primary_person; current_block_primary_is_female = True
                else:
                    current_block_primary_person = primary_name_phrase 
                    current_block_primary_is_female = (current_block_primary_relation_desc in ["पतोह", "पतोहू", "बेटी", "लड़की", "पत्नी"])
                    if current_block_father_name_of_primary and not current_block_primary_is_female:
                         current_block_primary_is_female = False
            print(f"  NEW PRIMARY (Final Check): '{current_block_primary_person}' (Desc: '{current_block_primary_relation_desc or 'स्वयं'}', Gender_is_female: {current_block_primary_is_female})")
            
            if current_block_primary_person:
                gender_primary = "स्त्री" if current_block_primary_is_female else "पुरुष"
                relation_primary_str = "स्वयं" 
                if current_block_primary_is_female and current_block_husband_name:
                    relation_primary_str = f"{current_block_husband_name} की पत्नी"
                    if current_block_primary_relation_desc and current_block_primary_relation_desc not in ["स्वयं", "पत्नी", ""]: relation_primary_str += f" ({current_block_primary_relation_desc})"
                elif not current_block_primary_is_female and current_block_wife_name: relation_primary_str = f"{current_block_wife_name} के पति"
                elif current_block_father_name_of_primary and not current_block_primary_is_female : relation_primary_str = f"{current_block_father_name_of_primary} का बेटा"
                elif current_block_grandfather_name_of_primary and not current_block_father_name_of_primary and not current_block_primary_is_female: relation_primary_str = f"{current_block_grandfather_name_of_primary} का पोता/नाती"
                elif current_block_primary_relation_desc and current_block_primary_relation_desc not in ["स्वयं", ""]: relation_primary_str = f"{current_block_primary_relation_desc} (स्वयं)"
                
                print(f"    DEBUG: Preparing to add Primary: {current_block_primary_person}, Rel: {relation_primary_str}")
                individuals_for_this_line.append({"Given Name": current_block_primary_person, "Relation": relation_primary_str, "Gender": gender_primary, "_is_block_primary": True})
                added_names_on_this_line.add((current_block_primary_person.lower(), relation_primary_str))
                print(f"    DEBUG: Added Primary to individuals_for_this_line: {current_block_primary_person}")

                if current_block_husband_name and current_block_primary_is_female: 
                    husband_rel_str = f"{current_block_primary_person} के पति"
                    if not any(current_block_husband_name.lower() == ar[0] and ar[1] == husband_rel_str for ar in added_names_on_this_line):
                        individuals_for_this_line.append({"Given Name": current_block_husband_name, "Relation": husband_rel_str, "Gender": "पुरुष", "_is_block_primary": False}); added_names_on_this_line.add((current_block_husband_name.lower(), husband_rel_str))
                elif current_block_wife_name and not current_block_primary_is_female: 
                    wife_rel_str = f"{current_block_primary_person} की पत्नी"
                    if not any(current_block_wife_name.lower() == ar[0] and ar[1] == wife_rel_str for ar in added_names_on_this_line):
                        individuals_for_this_line.append({"Given Name": current_block_wife_name, "Relation": wife_rel_str, "Gender": "स्त्री", "_is_block_primary": False}); added_names_on_this_line.add((current_block_wife_name.lower(), wife_rel_str))
                
                if current_block_father_name_of_primary:
                    father_rel_str_for_list = f"पिता ({current_block_primary_person} का)"
                    if not any(current_block_father_name_of_primary.lower() == ar[0] and ar[1] == father_rel_str_for_list for ar in added_names_on_this_line):
                        individuals_for_this_line.append({"Given Name": current_block_father_name_of_primary, "Relation": father_rel_str_for_list, "Gender": "पुरुष", "_is_block_primary": False}); added_names_on_this_line.add((current_block_father_name_of_primary.lower(), father_rel_str_for_list))
                elif current_block_grandfather_name_of_primary: 
                    gf_rel_str_for_list = f"दादा ({current_block_primary_person} का)"
                    if not any(current_block_grandfather_name_of_primary.lower() == ar[0] and ar[1] == gf_rel_str_for_list for ar in added_names_on_this_line):
                         individuals_for_this_line.append({"Given Name": current_block_grandfather_name_of_primary, "Relation": gf_rel_str_for_list, "Gender": "पुरुष", "_is_block_primary": False}); added_names_on_this_line.add((current_block_grandfather_name_of_primary.lower(), gf_rel_str_for_list))
            
            text_to_parse_for_current_line = text_after_primary_phrase_on_line 
            print(f"    DEBUG: Text set for iterative parsing (after primary processing): '{text_to_parse_for_current_line[:70]}'")
            
        # --- Iterative Relation Parsing ---
        if current_block_primary_person: 
            print(f"    ENTERING ITERATIVE RELATION PARSING for '{current_block_primary_person}'. Text: '{text_to_parse_for_current_line[:70]}'")
            active_subject_for_relations = current_block_primary_person 
            iteration_limit = 15; iter_count = 0
            while text_to_parse_for_current_line.strip() and iter_count < iteration_limit:
                iter_count += 1; text_changed_in_this_iteration = False
                text_to_parse_for_current_line = text_to_parse_for_current_line.lstrip(", ").strip()
                if not text_to_parse_for_current_line: print(f"      Iterative Pass {iter_count}. Text became empty. Breaking."); break
                print(f"      Iterative Pass {iter_count}. Text: '{text_to_parse_for_current_line[:70]}'")
                
                pattern_categories_order = [
                    ("SELF_RELATION_TO_ANCESTOR_PATTERNS", SELF_RELATION_TO_ANCESTOR_PATTERNS),
                    ("OWNER_OF_RELATION_PATTERNS", OWNER_OF_RELATION_PATTERNS),
                    ("OWNER_KE_RELATION_PATTERNS", OWNER_KE_RELATION_PATTERNS), 
                    ("IMPLIED_SUBJECT_RELATIONS (ke/ki)", IMPLIED_SUBJECT_RELATIONS),
                    ("IMPLIED_SUBJECT_DIRECT_RELATIONS", IMPLIED_SUBJECT_DIRECT_RELATIONS),
                    ("EXPLICIT_PAIR_RELATIONS", EXPLICIT_PAIR_RELATIONS)
                ]

                for category_name, pattern_list in pattern_categories_order:
                    # print(f"        Trying {category_name} on: '{text_to_parse_for_current_line[:70]}'") # Can be verbose
                    for rel_keyword, pattern in pattern_list:
                        if rel_keyword == "परदादा" and text_to_parse_for_current_line.startswith("पो- रामप्रसाद"):
                            print(f"          DEBUG P4-LIKE (SELF_RELATION_TO_ANCESTOR): Trying pattern '{pattern.pattern}' on repr: {repr(text_to_parse_for_current_line[:50])}")
                        
                        match_obj = pattern.match(text_to_parse_for_current_line) if category_name not in ["OWNER_OF_RELATION_PATTERNS", "OWNER_KE_RELATION_PATTERNS"] else pattern.search(text_to_parse_for_current_line)
                        
                        if match_obj:
                            print(f"          {category_name} Matched '{rel_keyword}' with '{match_obj.group(0)[:50]}'")
                            names_to_process_as_related = []
                            owner_for_relation = active_subject_for_relations 
                            relation_type_for_this_match = rel_keyword
                            gender_for_this_match = "पुरुष" if rel_keyword in ["भाई", "लड़का", "बेटा", "पिता", "दादा", "पोता", "मामा", "परदादा"] else \
                                                    "स्त्री" if rel_keyword in ["बेटी", "लड़की", "पत्नी", "पतोह", "पतोहू", "माता"] else ""

                            if category_name == "SELF_RELATION_TO_ANCESTOR_PATTERNS":
                                ancestor_name_raw = match_obj.group(1).strip() if pattern.groups > 0 and len(match_obj.groups()) >=1 and match_obj.group(1) else ""
                                names_to_process_as_related = tokenize_complex_name(ancestor_name_raw)
                            elif category_name in ["OWNER_OF_RELATION_PATTERNS", "OWNER_KE_RELATION_PATTERNS"]:
                                owner_name_raw = match_obj.group(1).strip(); related_text_chunk = match_obj.group(2).strip()
                                owner_name = clean_extracted_name(owner_name_raw)
                                if owner_name and owner_name.lower() != active_subject_for_relations.lower() and \
                                   not any(owner_name.lower() == ar[0] and ar[1] == "अभिभावक" for ar in added_names_on_this_line):
                                    individuals_for_this_line.append({"Given Name": owner_name, "Relation": "अभिभावक", "Gender": "पुरुष", "_is_block_primary": False})
                                    added_names_on_this_line.add((owner_name.lower(), "अभिभावक"))
                                owner_for_relation = owner_name or "अज्ञात"
                                names_to_process_as_related = split_conjoined_names(related_text_chunk)
                            elif category_name in ["IMPLIED_SUBJECT_RELATIONS (ke/ki)", "IMPLIED_SUBJECT_DIRECT_RELATIONS"]:
                                related_names_chunk_or_name = match_obj.group(1).strip()
                                if pattern.pattern.endswith("({_MULTI_WORD_NAME_REGEX})") or pattern.pattern.endswith("({_MULTI_WORD_NAME_REGEX)})"):
                                     names_to_process_as_related.append(related_names_chunk_or_name)
                                else: 
                                    names_to_process_as_related = split_conjoined_names(related_names_chunk_or_name)
                            elif category_name == "EXPLICIT_PAIR_RELATIONS":
                                person1_name = clean_extracted_name(match_obj.group(1)); person2_name = clean_extracted_name(match_obj.group(2))
                                if person1_name and person2_name:
                                    rel_p1 = "अभिभावक"; rel_p2 = f"{rel_keyword} ({person1_name} का)"
                                    gender_p1 = "पुरुष"; gender_p2 = "पुरुष" if rel_keyword in ["लड़का", "बेटा", "भाई"] else "स्त्री"
                                    if rel_keyword in ["लड़का", "बेटा"]: rel_p1 = f"पिता ({person2_name} का)"
                                    elif rel_keyword == "भाई": rel_p1 = f"भाई ({person2_name} का)"; rel_p2 = f"भाई ({person1_name} का)"
                                    if not any(person1_name.lower() == ar[0] and ar[1] == rel_p1 for ar in added_names_on_this_line):
                                        individuals_for_this_line.append({"Given Name": person1_name, "Relation": rel_p1, "Gender": gender_p1, "_is_block_primary": False}); added_names_on_this_line.add((person1_name.lower(), rel_p1))
                                    if not any(person2_name.lower() == ar[0] and ar[1] == rel_p2 for ar in added_names_on_this_line):
                                        individuals_for_this_line.append({"Given Name": person2_name, "Relation": rel_p2, "Gender": gender_p2, "_is_block_primary": False}); added_names_on_this_line.add((person2_name.lower(), rel_p2))
                            
                            for r_name_str in names_to_process_as_related:
                                tok_r_names = tokenize_complex_name(r_name_str)
                                for cn_r_name in tok_r_names:
                                    fin_cn_r_name = clean_extracted_name(cn_r_name)
                                    if not fin_cn_r_name or len(fin_cn_r_name) < 2 or fin_cn_r_name.lower() in INVALID_NAME_PARTICLES_AS_NAMES or fin_cn_r_name.isdigit():
                                        if fin_cn_r_name: print(f"            Skipping iteratively extracted name '{fin_cn_r_name}' due to filter.")
                                        continue
                                    current_relation_str = f"{relation_type_for_this_match} ({owner_for_relation} का/की)"
                                    if fin_cn_r_name.lower() == active_subject_for_relations.lower() and owner_for_relation == active_subject_for_relations : continue
                                    if not any(fin_cn_r_name.lower() == ar[0] and ar[1] == current_relation_str for ar in added_names_on_this_line):
                                        individuals_for_this_line.append({"Given Name": fin_cn_r_name, "Relation": current_relation_str, "Gender": gender_for_this_match, "_is_block_primary": False})
                                        added_names_on_this_line.add((fin_cn_r_name.lower(), current_relation_str))
                            text_to_parse_for_current_line = text_to_parse_for_current_line[match_obj.end():].strip()
                            text_changed_in_this_iteration = True; break 
                    if text_changed_in_this_iteration: break 
                if text_changed_in_this_iteration: print(f"        Text after {category_name}: '{text_to_parse_for_current_line[:70]}'"); continue
                if not text_changed_in_this_iteration: 
                    print(f"    Iterative relation parsing: No change in text for pass {iter_count}. Breaking. Text: '{text_to_parse_for_current_line[:70]}'")
                    break 
            print(f"    Finished iterative relation parsing. Remaining text: '{text_to_parse_for_current_line[:70]}'")
        
        final_remaining_text_on_line = text_to_parse_for_current_line.strip()
        
        # --- Date, Folio, and Orphan processing ---
        original_date_text_for_field_on_line = "" 
        m_ddmmyyyy_dinank = DATE_DDMM_YYYY_FORMAT_REGEX.search(final_remaining_text_on_line)
        m_dd_mm_yy = DATE_DD_MM_YY_FORMAT_REGEX.search(final_remaining_text_on_line)

        if m_ddmmyyyy_dinank:
            original_date_text_for_field_on_line = m_ddmmyyyy_dinank.group(0).strip()
            print(f"  Date found on this line (DDMM/YYYY format): '{original_date_text_for_field_on_line}'")
            current_block_date_text = original_date_text_for_field_on_line 
            current_block_date_gregorian = "" 
            try:
                day_d = int(m_ddmmyyyy_dinank.group('day')); month_d = int(m_ddmmyyyy_dinank.group('month')); year_d = int(m_ddmmyyyy_dinank.group('year'))
                if 1 <= day_d <= 31 and 1 <= month_d <= 12 and 1900 <= year_d <= 2300: 
                     current_block_date_gregorian = f"{year_d:04d}-{month_d:02d}-{day_d:02d}"
                     print(f"    Parsed as Gregorian: {current_block_date_gregorian}")
                else: print(f"    Cannot parse '{original_date_text_for_field_on_line}' as valid Gregorian date (day/month/year out of range).")
            except ValueError: print(f"    Error parsing DDMM/YYYY date: {original_date_text_for_field_on_line}")
            final_remaining_text_on_line = final_remaining_text_on_line.replace(original_date_text_for_field_on_line, "", 1).strip()
        elif m_dd_mm_yy: 
            original_date_text_for_field_on_line = m_dd_mm_yy.group(0).strip()
            print(f"  Date found on this line (DD/MM/YY format): '{original_date_text_for_field_on_line}'")
            current_block_date_text = original_date_text_for_field_on_line 
            current_block_date_gregorian = ""
            try:
                day_d = int(m_dd_mm_yy.group('day')); month_d = int(m_dd_mm_yy.group('month')); year_d_str = m_dd_mm_yy.group('year'); year_d = int(year_d_str)
                if len(year_d_str) == 2: year_d += 2000 
                if 1 <= day_d <= 31 and 1 <= month_d <= 12 and 1900 <= year_d <= 2300:
                     current_block_date_gregorian = f"{year_d:04d}-{month_d:02d}-{day_d:02d}"
                     print(f"    Parsed as Gregorian: {current_block_date_gregorian}")
                else: print(f"    Cannot parse '{original_date_text_for_field_on_line}' as valid Gregorian date.")
            except ValueError: print(f"    Error parsing DD/MM/YY date: {original_date_text_for_field_on_line}")
            final_remaining_text_on_line = final_remaining_text_on_line.replace(original_date_text_for_field_on_line, "", 1).strip()
        
        m_std_date = STANDARD_DATE_REGEX.search(final_remaining_text_on_line)
        if not original_date_text_for_field_on_line and m_std_date: 
            original_date_text_for_field_on_line = m_std_date.group(0).strip()
            hindu_month_str = m_std_date.group('month_name'); temp_block_month = hindu_month_str 
            hindu_day_str = m_std_date.group('day'); temp_block_day = hindu_day_str
            vs_year_str = m_std_date.group('year'); temp_block_year = vs_year_str
            paksha_str = m_std_date.group('paksha') 
            if paksha_str: temp_block_paksha = paksha_str
            if not hindu_day_str and not vs_year_str and not paksha_str: 
                if original_date_text_for_field_on_line.replace("मिती","").replace("मिटी","").strip() == hindu_month_str:
                    print(f"  Month Mention: '{original_date_text_for_field_on_line}' (Storing '{hindu_month_str}')")
                    final_remaining_text_on_line = final_remaining_text_on_line.replace(original_date_text_for_field_on_line, "", 1).strip(); original_date_text_for_field_on_line = ""
            elif hindu_day_str and hindu_month_str and vs_year_str and paksha_str: 
                current_block_date_text = original_date_text_for_field_on_line
                greg_date = convert_hindu_to_gregorian_date(hc_instance, hindu_day_str, f"{hindu_month_str} {paksha_str}", vs_year_str)
                current_block_date_gregorian = greg_date or ""
                print(f"  Date found on this line (sets block context): '{current_block_date_text}' -> '{current_block_date_gregorian}'")
                final_remaining_text_on_line = final_remaining_text_on_line.replace(original_date_text_for_field_on_line, "", 1).strip()
                temp_block_month, temp_block_paksha, temp_block_day, temp_block_year = None, None, None, None 
            else: 
                print(f"  Partial Date found (from std regex): '{original_date_text_for_field_on_line}' (Day:{hindu_day_str}, Month:{hindu_month_str}, Year:{vs_year_str}, Paksha:{paksha_str})")
                if original_date_text_for_field_on_line: 
                     final_remaining_text_on_line = final_remaining_text_on_line.replace(original_date_text_for_field_on_line, "", 1).strip()
        
        if not original_date_text_for_field_on_line or (m_std_date and not (m_std_date.group('day') and m_std_date.group('year') and m_std_date.group('paksha'))):
            m_paksha_tithi = PAKSHA_TITHI_REGEX.match(final_remaining_text_on_line)
            if m_paksha_tithi:
                paksha_tithi_text = m_paksha_tithi.group(0).strip()
                original_date_text_for_field_on_line = paksha_tithi_text 
                temp_block_paksha = m_paksha_tithi.group(1)
                day_candidate = m_paksha_tithi.group(2)
                if day_candidate in ["च", "छ"]: temp_block_day = "8" 
                else: temp_block_day = day_candidate
                print(f"  Partial Date (Paksha/Tithi only): '{paksha_tithi_text}' -> Paksha: {temp_block_paksha}, Day: {temp_block_day}")
                final_remaining_text_on_line = final_remaining_text_on_line.replace(paksha_tithi_text, "", 1).strip()
        
        if not original_date_text_for_field_on_line and not temp_block_year and not current_block_date_gregorian : 
            m_year_standalone = re.search(r"((?:सम्बत्?|सम्बर|सं\.?)\s*\d{4,5}(?:इस|स)?)", final_remaining_text_on_line) 
            if m_year_standalone: 
                year_text_standalone = m_year_standalone.group(1).strip()
                original_date_text_for_field_on_line = year_text_standalone
                year_match_in_standalone = re.search(r"(\d{4,5})", year_text_standalone) 
                if year_match_in_standalone: temp_block_year = year_match_in_standalone.group(1)
                if not temp_block_month and not temp_block_day and not temp_block_paksha: 
                    current_block_date_text = year_text_standalone; current_block_date_gregorian = ""
                    print(f"  Date found on this line (Year only, sets block context): '{current_block_date_text}'")
                else: 
                    print(f"  Year Mention: '{year_text_standalone}' (Storing year: '{temp_block_year}')")
                final_remaining_text_on_line = final_remaining_text_on_line.replace(year_text_standalone, "", 1).strip()

        if temp_block_month and temp_block_day and temp_block_year and temp_block_paksha and \
           (not current_block_date_gregorian or original_date_text_for_field_on_line): 
            assembled_date_str = f"{temp_block_month} {temp_block_paksha} {temp_block_day} सं. {temp_block_year}"
            current_block_date_text = assembled_date_str
            greg_date = convert_hindu_to_gregorian_date(hc_instance, temp_block_day, f"{temp_block_month} {temp_block_paksha}", temp_block_year)
            current_block_date_gregorian = greg_date or ""
            print(f"  Assembled Full Date for Block: '{current_block_date_text}' -> Gregorian: '{current_block_date_gregorian}'")
            temp_block_month, temp_block_paksha, temp_block_day, temp_block_year = None, None, None, None

        m_folio_para = re.search(r'(पद्मा|पन्ना|पत्र)\s*(\d+)', final_remaining_text_on_line, re.IGNORECASE)
        if m_folio_para: 
            line_folio_number_temp = f"{m_folio_para.group(1)} {m_folio_para.group(2)}"
            final_remaining_text_on_line = final_remaining_text_on_line.replace(m_folio_para.group(0),"",1).strip()
            current_block_folio_number = line_folio_number_temp 
            print(f"  Folio found on this line (sets block context): '{current_block_folio_number}'")
        
        # --- Record Creation ---
        print(f"    DEBUG: individuals_for_this_line before final record creation loop FOR PARA {para_idx+1}: {individuals_for_this_line}") 
        if individuals_for_this_line:
            for i, ind_data in enumerate(individuals_for_this_line):
                additional_info_for_record = ""
                if i == len(individuals_for_this_line) - 1: # Add remaining text to last individual from this line
                    additional_info_for_record = final_remaining_text_on_line

                given_name_full = ind_data.get("Given Name", "")
                surname_extracted = ""
                if len(given_name_full.split()) > 1:
                    last_word = given_name_full.split()[-1]
                    # More comprehensive list of common surnames/caste names
                    common_surnames = ["शर्मा", "सिंह", "कुमार", "पाठक", "जोशी", "शुक्ला", "महतो", "प्रसाद", "लाल", "यादव", "गुप्ता", "वर्मा", "देवी", "बाई", "तिवारी", "पाण्डेय", "मिश्रा", "चौधरी", "श्रीवास्तव", "खंगार", "राठौर", "तोमर", "सोदिया", "बघेले", "कुशवाहा"]
                    if last_word in common_surnames:
                        surname_extracted = last_word
                        given_name_full = " ".join(given_name_full.split()[:-1])

                record = {"File Name": file_name, "Bahi Number": bahi_number_global, 
                          "Data Position": para_data_position, 
                          "Caste": line_caste, "Subcaste": line_subcaste, "From Which Place": line_place, 
                          "Date of Ritual": current_block_date_text, "Date of Ritual (Gregorian)": current_block_date_gregorian, 
                          "Whose Ritual 1": ind_data.get("WhoseRitualOverride") or (current_block_primary_person if not ind_data.get("_is_block_primary") else ""), 
                          "Folio Number": current_block_folio_number, 
                          "Individual ID": f"P{overall_individual_id_counter:04d}", 
                          "Given Name": given_name_full, "Surname": surname_extracted,
                          "Relation": ind_data.get("Relation", ""), 
                          "Gender": ind_data.get("Gender", ""), 
                          "Family Id": family_id, 
                          "Additional Information 1": additional_info_for_record
                         }
                all_extracted_records.append(record); overall_individual_id_counter += 1
        elif final_remaining_text_on_line : 
            orphan_whose_ritual = current_block_primary_person or ""
            relation_for_orphan = "उल्लेखित (शेष)"
            if final_remaining_text_on_line.strip() == "ई-": relation_for_orphan = "उल्लेख (कोड)"
            record = {"File Name": file_name, "Bahi Number": bahi_number_global, 
                      "Data Position": para_data_position, 
                      "Caste": line_caste, "Subcaste": line_subcaste, "From Which Place": line_place, 
                      "Date of Ritual": current_block_date_text, "Date of Ritual (Gregorian)": current_block_date_gregorian, 
                      "Whose Ritual 1": orphan_whose_ritual, 
                      "Folio Number": current_block_folio_number, 
                      "Individual ID": f"P{overall_individual_id_counter:04d}", 
                      "Given Name": final_remaining_text_on_line.strip(), 
                      "Relation": relation_for_orphan, "Gender": "", "Family Id": family_id,
                      "Additional Information 1": final_remaining_text_on_line.strip()}
            all_extracted_records.append(record); overall_individual_id_counter += 1
            print(f"  Potential orphan from remaining text on line: '{final_remaining_text_on_line.strip()}'")
            
    return all_extracted_records

def main():
    image_path = "Downloads/Transcript_samples/Transcript_samples/image00015.jpg" 
    DEBUG_WITH_SAMPLE = False 
    TARGET_FILE_FOR_DEBUG = "00250" 
    credentials_path = "OneDrive/Desktop/round-cacao-459815-s8-9f2a78c8d027.json" 
    hindu_calendar_gist_dir = "Downloads/0090a0460608728f32381164ea54865c-1dcbafe4e559a8d8d471b7e4d46a5fcfed21fea6/0090a0460608728f32381164ea54865c-1dcbafe4e559a8d8d471b7e4d46a5fcfed21fea6" 
    
    if os.path.exists(credentials_path): os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credentials_path
    elif not os.getenv('GOOGLE_APPLICATION_CREDENTIALS'): print("⚠️ CRITICAL WARNING: GOOGLE_APPLICATION_CREDENTIALS not set.")
    
    global _HinduCalendar, HINDU_CALENDAR_AVAILABLE 
    if hindu_calendar_gist_dir and os.path.isdir(hindu_calendar_gist_dir):
        if hindu_calendar_gist_dir not in sys.path:
            sys.path.insert(0, hindu_calendar_gist_dir)
            if not HINDU_CALENDAR_AVAILABLE: 
                try:
                    from hindu_calendar import HinduCalendar as HC_retry
                    _HinduCalendar = HC_retry
                    HINDU_CALENDAR_AVAILABLE = True
                    print("HinduCalendar module re-loaded successfully after adding Gist path.")
                except ImportError: print(f"⚠️ WARNING: Still could not import HinduCalendar from {hindu_calendar_gist_dir}")
                except Exception as e: print(f"⚠️ WARNING: Error re-importing HinduCalendar: {e}")

    text_to_process = ""; file_name_to_process = ""
    if DEBUG_WITH_SAMPLE:
        pass
    else: 
        print(f"--- PRODUCTION MODE: Processing image {image_path} ---")
        if not os.path.exists(image_path): print(f"⚠️ ERROR: Image file not found at '{image_path}'."); return
        
        ocr_text_list = []
        try:
            ocr_text_list = detect_text(image_path)
        except Exception as e:
            print(f"💥 OCR DETECTION FAILED: {e}")
            return 
        if not ocr_text_list or not ocr_text_list[0]: print(f"⚠️ No text detected in {image_path}."); return
        
        raw_ocr_text = ocr_text_list[0]
        cleaned_text_with_markers = clean_text(raw_ocr_text) 
        print(f"\n--- Cleaned Full Text (Newlines Marked) --- \n{cleaned_text_with_markers}\n------------------------")
        text_to_process = cleaned_text_with_markers; file_name_to_process = os.path.basename(image_path)

    hc_instance = None
    if HINDU_CALENDAR_AVAILABLE and _HinduCalendar:
        try: 
            hc_instance = _HinduCalendar(method='hindi', city='auto') 
            print("Gist's HinduCalendar initialized.")
        except Exception as e: print(f"⚠️ Failed to initialize Gist's HinduCalendar: {e}")
    
    language_client = None
    try:
        if os.getenv('GOOGLE_APPLICATION_CREDENTIALS') and hasattr(language_v1, 'LanguageServiceClient') and callable(language_v1.LanguageServiceClient): # type: ignore
            language_client = language_v1.LanguageServiceClient() # type: ignore
            print("Natural Language API client initialized.")
        else: print("Natural Language API client not initialized (credentials, library missing, or stub in use).")
    except Exception as e: print(f"⚠️ Failed to initialize NLP API client: {e}.")
    
    records = process_document_structure(text_to_process, language_client, hc_instance, file_name=file_name_to_process)
    
    if not records: print(f"📋 No records extracted from {file_name_to_process}."); return
    
    df = pd.DataFrame(records)
    
    desired_columns = [
        'Image No', 'Panda Name', 'Bahi Name', 'Bahi Number', 'Folio Number', 
        'Data Position', 'District', 'Tehasil', 'Station', 'Postoffice', 
        'City/Village', 'From Which Place', 
        'Caste', 'Subcaste', 'Individual ID', 'Given Name', 'Surname', 'Relation', 
        'Gender', 'Family Id', 'Ritual Name', 'Date of Ritual', 
        'Date of Ritual (Gregorian)', 
        'Date of Ritual 2',
        'Whose Ritual 1', 'Whose Ritual 2', 'Contact Number 1', 'Contact Number 2',
        'Flags', 'Exceptions and remarks', 
        'Additional Information 1', 'Additional Information 2', 
        'Additional Information 3', 'Additional Information 4'
    ]
    
    image_no_val = ""
    img_no_match = re.search(r'(\d+)', file_name_to_process)
    if img_no_match:
        image_no_val = img_no_match.group(1)
    
    for col in desired_columns: 
        if col not in df.columns: 
            if col == 'Image No':
                df[col] = image_no_val if image_no_val else file_name_to_process 
            elif col == 'Data Position': 
                 if 'Data Position (Paragraph)' in df.columns:
                    df.rename(columns={'Data Position (Paragraph)': 'Data Position'}, inplace=True)
                 elif 'Data Position' not in df.columns: 
                    df[col] = "" 
            else: 
                df[col] = "" 
    
    final_df_columns = [col for col in desired_columns if col in df.columns] 
    df = df[final_df_columns]
    
    print(f"\n📋 Extracted Data from {file_name_to_process}:")
    pd.set_option('display.max_columns', None); pd.set_option('display.expand_frame_repr', False)
    print(df.to_string(index=False))
    
    base_name, _ = os.path.splitext(file_name_to_process)
    output_file_suffix = f"test_{TARGET_FILE_FOR_DEBUG}" if DEBUG_WITH_SAMPLE else "prod_image"
    output_dir = "output_extractions"; 
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    output_file = os.path.join(output_dir, f"{base_name}_extracted_v60.25_{output_file_suffix}.xlsx") 
    try: 
        df.to_excel(output_file, index=False, engine='openpyxl')
        print(f"\n✅ Data saved to '{output_file}'")
    except Exception as e: print(f"⚠️ Error saving data to Excel: {e}")

if __name__ == "__main__":
    main()


⚠️ WARNING: hindu_calendar.py not found.
--- PRODUCTION MODE: Processing image Downloads/Transcript_samples/Transcript_samples/image00015.jpg ---

--- Cleaned Full Text (Newlines Marked) --- 
क###NEWLINE###शासनादपमरेवावाला प्रारोज मोचतरा मनर###NEWLINE###टाडेर्जनमन्दी के पास पाचकेशकुमार शर्मा बेटा स्वयूरेशमजी के###NEWLINE###पो- रामप्रसाद के भाई श्री मनोहरलालक्ष्मी लड़का संगणनूषमान###NEWLINE###पन्समा - चोचास्व. रघुवीर प्रसाद कापेटा राज कुमार-जीतेन्द्र-###NEWLINE###मोहून- संगमीदीपीका शर्मा संग जीद कमलेश शर्मा क रहीपाटेक###NEWLINE###वाले चपता का सम्पत मार दी###NEWLINE###दिनांक 3109/2287###NEWLINE###काचारेक // ग्राणसमाद्यगोष###NEWLINE###// ग्रावसगादयगोणस खोपासी गरीन मोनारायमत्यालोमी###NEWLINE###वेतनयमाप्रसाद के पो. नन्दलाल के भावनेश शर्मा###NEWLINE###कमलेश###NEWLINE###9###NEWLINE###कान्तुरानी- सन्तोष रामो लडप्यासत्यम् शिवम् संगसाले चक्रेशकुमार###NEWLINE###प्रशोन वालों के सम्वत 2004 भादीबदी 7###NEWLINE###कमलेश शर्मा###NEWLINE###थ###NEWLINE###मुख्यगविशष्ठदेवनासा भेदासाला. आपन मो- छतरा###NEWLIN